# VPP Dispatch — Multi-Asset Fleet

This notebook builds a `CustomerConfig` with an explicit fleet — PV, battery, a **continuous** flexible load, a fixed load, and a grid connection — and runs `run_multi_asset_dispatch`.

It also demonstrates the two other flexible-load modes (`on_off`, `shiftable`) in isolation.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.vpp_dispatch.models.schemas import CustomerConfig
from src.vpp_dispatch.services.dispatch_service import run_multi_asset_dispatch, create_optimization_summary

## 1. Configure the fleet

In [ ]:
T = 24
hours = np.arange(T)
pv_kw = np.clip(6 * np.sin((hours - 6) / 12 * np.pi), 0, None).round(2).tolist()
price_buy = (0.15 + 0.35 * (np.sin((hours - 18) / 24 * 2 * np.pi) ** 4)).round(3).tolist()
price_sell = [0.05] * T

config = CustomerConfig(
    customer_id='multi_asset_demo',
    time_periods=T,
    assets=[
        {'asset_id': 'pv_1', 'asset_type': 'pv', 'pv_profile_kw': pv_kw},
        {
            'asset_id': 'battery_1', 'asset_type': 'battery',
            'capacity_kwh': 10, 'p_charge_max_kw': 5, 'p_discharge_max_kw': 5,
            'soc_initial': 3.0, 'eff_charge': 0.92, 'eff_discharge': 0.92,
        },
        {
            'asset_id': 'flex_1', 'asset_type': 'flex_load',
            'is_continuous': True, 'p_min_kw': 0.0, 'p_max_kw': 3.0,
            'energy_required_kwh': 8.0, 'time_window': [0, T - 1],
        },
        {'asset_id': 'fixed_1', 'asset_type': 'fixed_load', 'fixed_load_profile_kw': [1.5] * T},
        {
            'asset_id': 'grid_1', 'asset_type': 'grid',
            'import_max_kw': 50, 'export_max_kw': 50,
            'price_buy': price_buy, 'price_sell': price_sell,
        },
    ],
)

## 2. Run dispatch

In [ ]:
results, status = run_multi_asset_dispatch(config)
print(status)

summary = create_optimization_summary(results)
summary

## 3. Plot grid power vs. price

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(hours, results['p_grid'], alpha=0.6, label='Grid power (kW)')
ax2 = ax.twinx()
ax2.plot(hours, price_buy, color='red', label='Buy price')
ax.set_xlabel('Hour')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Grid import/export vs. price')
plt.show()

## 4. Per-asset results

In [ ]:
for asset_id, asset_data in results['assets'].items():
    print(f"\n--- {asset_data['type']} ({asset_id}) ---")
    for k, v in asset_data['results'].items():
        if isinstance(v, list):
            print(f'  {k}: [{min(v):.2f} .. {max(v):.2f}], sum={sum(v):.2f}')
        else:
            print(f'  {k}: {v}')

## 5. The three flexible-load modes, side by side

`FlexLoadAsset` supports three mutually-exclusive modes. Here each is dispatched alone against the same cheap/expensive price pattern to show how differently they behave.

In [ ]:
T2 = 6
price = [0.5, 0.1, 0.5, 0.1, 0.5, 0.5]
grid_asset = {
    'asset_id': 'grid_1', 'asset_type': 'grid',
    'import_max_kw': 20, 'export_max_kw': 20,
    'price_buy': price, 'price_sell': [0.02] * T2,
}

modes = {
    'continuous': {
        'asset_id': 'flex_1', 'asset_type': 'flex_load', 'is_continuous': True,
        'p_min_kw': 0, 'p_max_kw': 4, 'energy_required_kwh': 6, 'time_window': [0, T2 - 1],
    },
    'on_off': {
        'asset_id': 'flex_1', 'asset_type': 'flex_load', 'is_on_off': True,
        'p_on_kw': 3, 'energy_required_kwh': 6, 'time_window': [0, T2 - 1],
    },
    'shiftable': {
        'asset_id': 'flex_1', 'asset_type': 'flex_load', 'is_shiftable': True,
        'load_profile': [2, 2, 2], 'time_window': [0, T2 - 1],
    },
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, (name, asset) in zip(axes, modes.items()):
    cfg = CustomerConfig(customer_id=f'flex_{name}', time_periods=T2, assets=[asset, grid_asset])
    res, st = run_multi_asset_dispatch(cfg)
    flex_power = res['assets']['flex_1']['results']['flex_power_kw']
    ax.bar(range(T2), flex_power)
    ax.set_title(f'{name} (solved: {st["success"]})')
    ax.set_xlabel('Period')
axes[0].set_ylabel('Flex load power (kW)')
plt.tight_layout()
plt.show()